# 03 — DDIM: Denoising Diffusion Implicit Models

DDIM achieves deterministic, fast sampling by treating the reverse process as a non-Markovian path.

## Deterministic Sampling and Fewer Steps

DDIM (Song et al., 2020) observes that the DDPM training objective only requires the marginals $q(x_t|x_0)$, not the full joint Markov chain. This means we can define a *different*, non-Markovian reverse process that shares the same marginals but allows:
1. **Deterministic sampling** (σ=0): given the same initial noise, DDIM always produces the same output
2. **Arbitrary step skipping**: instead of T=1000 reverse steps, we can use 10-50 steps by skipping along a subsequence

The DDIM update rule:
$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \underbrace{\frac{x_t - \sqrt{1-\bar{\alpha}_t}\epsilon_\theta}{\sqrt{\bar{\alpha}_t}}}_{\text{pred } x_0} + \sqrt{1-\bar{\alpha}_{t-1} - \sigma_t^2} \cdot \epsilon_\theta + \sigma_t \epsilon$$

When σ=0, the update is deterministic. Setting σ=√(β_t) recovers DDPM.

**Why fewer steps work**: DDIM defines a straight-line trajectory in data space, so each step makes larger progress. DDPM's Markovian chain takes small steps; skipping them introduces drift. DDIM's ODE formulation means the path curvature is low, making large steps accurate.

In [ ]:
# DDIM sampler — building on the trained DDPM noise_net
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Rebuild the model and data from NB02 (self-contained)
torch.manual_seed(42)
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

n = 2000
x_data = torch.cat([torch.randn(n//2)*0.3+2, torch.randn(n//2)*0.3-2]).unsqueeze(1)

class NoiseNet(nn.Module):
    def __init__(self, data_dim=1, hidden=64, T=200):
        super().__init__()
        self.time_embed = nn.Embedding(T, hidden)
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, data_dim),
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, self.time_embed(t)], dim=-1))

noise_net = NoiseNet()
opt = torch.optim.Adam(noise_net.parameters(), lr=3e-4)

def q_sample(x0, t):
    noise = torch.randn_like(x0)
    return alpha_bars[t].sqrt() * x0 + (1-alpha_bars[t]).sqrt() * noise, noise

for epoch in range(1500):
    idx = torch.randint(0, len(x_data), (256,))
    t = torch.randint(0, T, (256,))
    x_t, eps = q_sample(x_data[idx], t)
    loss = ((noise_net(x_t, t) - eps)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

print('Model retrained')

In [ ]:
# DDIM sampler
@torch.no_grad()
def ddim_sample(model, n_samples, alpha_bars, n_steps=20, eta=0.0):
    """
    eta=0: deterministic DDIM; eta=1: stochastic (DDPM-like)
    """
    # Subsequence of time steps
    step_size = len(alpha_bars) // n_steps
    timesteps = list(reversed(range(0, len(alpha_bars), step_size)))

    x = torch.randn(n_samples, 1)

    for i, t in enumerate(timesteps):
        t_batch = torch.full((n_samples,), t, dtype=torch.long)
        eps = model(x, t_batch)

        ab_t = alpha_bars[t]
        ab_prev = alpha_bars[timesteps[i+1]] if i+1 < len(timesteps) else torch.tensor(1.0)

        # Predicted x0
        x0_pred = (x - (1-ab_t).sqrt() * eps) / ab_t.sqrt()
        x0_pred = x0_pred.clamp(-3, 3)

        # Direction pointing to xt
        sigma = eta * ((1 - ab_prev) / (1 - ab_t) * (1 - ab_t / ab_prev)).sqrt()
        dir_xt = (1 - ab_prev - sigma**2).sqrt() * eps

        # Noise term
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        x = ab_prev.sqrt() * x0_pred + dir_xt + sigma * noise

    return x

# Compare DDPM (200 steps) vs DDIM (20 steps)
@torch.no_grad()
def ddpm_sample(model, n_samples):
    x = torch.randn(n_samples, 1)
    for t in range(T - 1, -1, -1):
        t_batch = torch.full((n_samples,), t, dtype=torch.long)
        eps = model(x, t_batch)
        mean = (1/alphas[t].sqrt()) * (x - betas[t]/(1-alpha_bars[t]).sqrt() * eps)
        if t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x

samples_ddpm = ddpm_sample(noise_net, 500)
samples_ddim = ddim_sample(noise_net, 500, alpha_bars, n_steps=20)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].hist(x_data.squeeze().numpy(), bins=40, density=True, alpha=0.7)
axes[0].set_title('Real data')
axes[1].hist(samples_ddpm.squeeze().numpy(), bins=40, density=True, alpha=0.7, color='orange')
axes[1].set_title('DDPM (200 steps)')
axes[2].hist(samples_ddim.squeeze().numpy(), bins=40, density=True, alpha=0.7, color='green')
axes[2].set_title('DDIM (20 steps = 10x speedup)')
for ax in axes: ax.set_xlim(-5, 5)
plt.suptitle('DDIM: 10x Speedup vs DDPM')
plt.tight_layout()
plt.savefig('/tmp/ddim_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('DDIM comparison saved')